# IBKR API notebook for fetching historical market data

#### Prerequisites: 
- ```pip install -r requirements.txt``` to install the necessary packages.

- Launch Trader Workstation (TWS) and enable ActiveX API (```File->Global Configuration->API->Settings``` then check ```Enable ActiveX and Socket Clients``` and uncheck ```Read-Only API```. Do not forget to apply the settings).

#### 1. Connection to IBKR API local gateway

```File->Global Configuration->API->Settings```
et cocher 
```Enable ActiveX and Socket Clients```
et décocher 
```Read-Only API```

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')
import pandas as pd
from ib_async import *
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.DEBUG)

## Request Historical data

#### 2. Choose your contract

### List of contract to fetch : 

**Forex**:
- EURUSD : `Forex(pair='EURUSD', exchange='IDEALPRO')`
- GBPUSD : `Forex(pair='GBPUSD', exchange='IDEALPRO')`
- USDJPY : `Forex(pair='USDJPY', exchange='IDEALPRO')` 
- CHFUSD : `Forex(pair='CHFUSD', exchange='IDEALPRO')`
- AUDUSD : `Forex(pair='AUDUSD',
  
**Stocks**:
- Apple Inc. : `Stock(symbol='AAPL', exchange='SMART', currency='USD')`

**Indices**:
- Nasdaq 100 : `Index('NDX', 'NASDAQ', 'USD')`
- S&P 500 : `Index('SPX', 'CBOE', 'USD')`
- Dow Jones : `Index(symbol='DD', exchange='CBOT', currency='USD')` (NOT WORKING)
- CAC40 : `Index('CAC40', 'MONEP', 'EUR')`
- DAX : `Index('DAX', 'EUREX', 'EUR')`
- NIKAI : `Index(symbol='N225', exchange='OSE.JPN', currency='JPY')`

**Futures**:
- Gold Futures December 2025 : `CFD(symbol='XAUUSD', exchange='SMART', currency='USD', lastTradeDateOrContractMonth='202512')`
- Petrol WTI : `CFD(symbol='IBUSOIL', exchange='SMART', currency='USD')`
- Petrol brent : `CFD(symbol='', )
- Argent : `CFD('XAGUSD' 'SMART', 'USD')`
- Gaz naturel : `Future(symbol='NG', exchange='NYMEX', currency='USD', lastTradeDateOrContractMonth='202512')`

**Cryptos**:
- Bitcoin : `Crypto('BTC', 'USD', 'PAXOS')`
- Ethereum : `Crypto('ETH', 'USD', 'PAXOS')`
- Ripple : `Crypto('XRP', 'USD', 'PAXOS')`
- Litecoin : `Crypto('LTC', 'USD', 'PAXOS')`


In [ ]:
# Define the contract HERE

# For futures, you need to specify either expiry or localSymbol
contract = CFD('IBUST100', 'SMART', 'USD')

ib.qualifyContracts(contract)
#======================================================================
# Below => just some printing on contract chosen:
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
        print()

#### (Optional) Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='ASK', useRTH=False) # ASK for Forex/CFD/Futures
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")
logging.info(f"Timestamp: {timestamp}")

#### 3. End date choice for data request

In [ ]:
# yesterday's date - UTC format for IBKR API
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d-%H:%M:%S')

In [ ]:
# today's date - UTC format for IBKR API  
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

In [ ]:
# custom end date - UTC format for IBKR API
# format: yyyymmdd-hh:mm:ss (UTC time, no timezone suffix needed)
end_date = '20250429-22:00:00'

#### 4. Main loop for fetching historical data

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`, etc...
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' # Candle period to fetch
request_duration = '3 M'  # Duration in days (use D, not "day"). Use a very big value if you want the maximum historical data, it will fetch the maximum available automatically.
price_source = 'ASK'  # 'BID', 'ASK', or 'TRADES' (note that for some symbols, (e.g. EURUSD) only 'BID' and 'ASK' are available)

bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

#### 5. Convert the list of bars to a data frame, print the first / last rows and remove useless columns:

In [ ]:
bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
# new_df = new_df.drop(columns=['volume', 'average', 'barCount'])
new_df = new_df.drop(columns=['average', 'barCount'])

# Display the updated DataFrame
new_df.head()

#### 6A. Save new DataFrame to CSV (only for no existing file)

**IMPORTANT**: Use the correct format file : `../marketData/{contract.symbol}_{candle_period}_{first_date}_to_{last_date}_{price_source}.csv`

Date format to use: `YYYYMMDD`

In [ ]:
# Determine first/last timestamps (works whether times are in a 'date' column or the index)
if 'date' in new_df.columns:
    start_date = pd.to_datetime(new_df['date'].iat[0]).strftime('%Y%m%d')
    end_date = pd.to_datetime(new_df['date'].iat[-1]).strftime('%Y%m%d')
save_path = f"../marketData/{contract.symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}_{price_source}.csv"

# Save the newly downloaded DataFrame
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Optional verification
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

#### 6B. DataFrame update (only for existing data)
Update the dataframe by merging the new data with old ones

In [ ]:
import pandas as pd
from Helpers import merge_ohlc_dataframes
import os

# Determine first/last timestamps (works whether times are in a 'date' column or the index)
if 'date' in new_df.columns:
    start_date = pd.to_datetime(new_df['date'].iat[0]).strftime('%Y%m%d')
    end_date = pd.to_datetime(new_df['date'].iat[-1]).strftime('%Y%m%d')

# Load your existing data - use index_col=0 to treat first column as index
existing_file_path = "../marketData/IBUST100_10secs_20250427_to_20250725_ASK.csv" # Here, enter the correct file path fo the existing csv data file
existing_df = pd.read_csv(existing_file_path, index_col=0)
display(existing_df.head())

# Merge the dataframes
merged_df = merge_ohlc_dataframes(existing_df, new_df, frequency='10s') # Adjust frequency as needed
display(merged_df.head())
display(merged_df.tail())

# Save the merged dataframe
save_path = f"../marketData/{contract.symbol}_10secs_{start_date}_to_{end_date}_{price_source}.csv" # Here, enter the correct file path for the new csv data file
merged_df.to_csv(save_path, index=True)
print(f"Merged data saved to: {save_path}")
# Delete the original file if needed
if os.path.exists(existing_file_path):
    os.remove(existing_file_path)
    print(f"Deleted original file: {existing_file_path}")


In [ ]:
import subprocess

cmd = [
    "scp",
    "marketData/IBUST100_10secs_20250323_to_20250429_ASK.csv",
    "maxime@192.168.1.100:/home/maxime/syncthing/data/FinTech/api_rest/marketData"
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("stdout:", result.stdout)
print("stderr:", result.stderr)

### Additional features 
- Checking data integrity

In [ ]:
from Helpers import checkDataFile, visualize_data_gaps
import matplotlib.pyplot as plt

symbol = 'GC'
interval = '10secs'
start_date = '20230321'
end_date = '20250829'
price_source= 'TRADES'

# 1. Load the existing dataframe
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.csv"

# Analyze data gaps
report = checkDataFile(
    file_path=save_path, 
    interval=interval
)

# Print summary
print(f"Analyzed {report['total_trading_days']} trading days")
print(f"Found {report['days_with_gaps']} days with gaps ({report['analysis_summary']['gap_percentage']:.2f}%)")
print(f"Total gaps detected: {report['total_gaps']}")

# Visualize the gaps
fig = visualize_data_gaps(report)
plt.show()

# To examine specific days with large gaps
problem_days = {date: data for date, data in report["gaps_by_date"].items() 
                if data["missing_points"] > 10}
print(f"Days with more than 10 missing points: {len(problem_days)}")
for date, data in sorted(problem_days.items()):
    print(f"{date}: Missing {data['missing_points']} of {data['expected_points']} points")